In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm


TEST_CSV = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"


DEBERTA_PATH = "microsoft/deberta-v3-small"
ROBERTA_PATH = "/kaggle/input/.../roberta"

In [12]:
import os
import shutil
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

# ────────────────────────────────────
# 1. LOAD DATA
# ────────────────────────────────────
train_df = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)
test_df = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)
OPTION_COLS = ["A", "B", "C", "D", "E"]
label2id = {opt: i for i, opt in enumerate(OPTION_COLS)}
id2label = {i: opt for opt, i in label2id.items()}

# ────────────────────────────────────
# 2. FORMAT TEXT
# ────────────────────────────────────
def format_question(row):
    prompt = row["prompt"]
    opts = [f"{col}) {row[col]}" for col in OPTION_COLS]
    return prompt + "\n" + "\n".join(opts)

train_df["text"] = train_df.apply(format_question, axis=1)
train_df["label"] = train_df["answer"].map(label2id)
test_df["text"] = test_df.apply(format_question, axis=1)

# ────────────────────────────────────
# 3. CORRECTED FINE‑TUNE FUNCTION
# ────────────────────────────────────
def fine_tune(model_name, train_df, output_dir, num_epochs=3):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=5
    )

    # Properly set pad token (both tokenizer and model config)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.eos_token_id
    else:
        model.config.pad_token_id = tokenizer.pad_token_id

    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=512,
        )

    dataset = Dataset.from_pandas(train_df[["text", "label"]])
    dataset = dataset.map(tokenize_fn, batched=True)
    dataset = dataset.rename_column("label", "labels")
    dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    split = dataset.train_test_split(test_size=0.1, seed=42)
    train_ds, val_ds = split["train"], split["test"]

    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=1e-5,            # lower LR for stability
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        fp16=False,                    # avoid fp16 issues
        gradient_checkpointing=False,  # keep full precision
        max_grad_norm=1.0,             # gradient clipping already default, but explicit
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        acc = accuracy_score(labels, preds)
        return {"accuracy": acc}

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return model, tokenizer

# ────────────────────────────────────
# 4. SAFE LOADING / RETRAINING
# ────────────────────────────────────
deberta_dir = "/kaggle/working/deberta_small_mcq"
roberta_dir = "/kaggle/working/roberta_base_mcq"

def is_valid_model_dir(directory):
    if not os.path.isdir(directory):
        return False
    for f in os.listdir(directory):
        if f.endswith(".safetensors") or f == "pytorch_model.bin":
            return True
    return False

def load_or_train(model_name, save_dir, force_retrain=False):
    if not force_retrain and is_valid_model_dir(save_dir):
        print(f"Loading fine‑tuned model from {save_dir}")
        tokenizer = AutoTokenizer.from_pretrained(save_dir)
        model = AutoModelForSequenceClassification.from_pretrained(save_dir)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            model.config.pad_token_id = tokenizer.eos_token_id
        return model, tokenizer
    else:
        if os.path.isdir(save_dir):
            print(f"Removing previous checkpoint {save_dir}")
            shutil.rmtree(save_dir)
        print(f"Fine‑tuning {model_name} ...")
        return fine_tune(model_name, train_df, save_dir, num_epochs=3)

# Force retrain both to ensure clean start
if is_valid_model_dir(deberta_dir):
    shutil.rmtree(deberta_dir)
if is_valid_model_dir(roberta_dir):
    shutil.rmtree(roberta_dir)

deberta_model, deberta_tokenizer = load_or_train(
    "microsoft/deberta-v3-small", deberta_dir, force_retrain=True
)
roberta_model, roberta_tokenizer = load_or_train(
    "roberta-base", roberta_dir, force_retrain=True
)

# ────────────────────────────────────
# 5. BATCHED INFERENCE (WITH NaN PROTECTION)
# ────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
deberta_model.to(device)
roberta_model.to(device)

def get_probabilities_batched(model, tokenizer, texts, batch_size=16):
    """Return softmax probabilities (n,5) using batching. Replace NaN with uniform."""
    model.eval()
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        # Check for NaN and replace with uniform distribution
        if np.any(np.isnan(probs)):
            nan_mask = np.isnan(probs).any(axis=-1)
            probs[nan_mask] = 1.0 / 5
        all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)

def top_k_from_probs(probs, k=3):
    order = np.argsort(-probs)
    return [id2label[i] for i in order[:k]]

# ────────────────────────────────────
# 6. MAP@3 HELPER
# ────────────────────────────────────
def mapk(actual, predicted, k=3):
    def apk(a, p):
        score, hits = 0.0, 0
        for i, pi in enumerate(p[:k]):
            if pi == a:
                hits += 1
                score += hits / (i + 1)
        return score
    return np.mean([apk(a, p) for a, p in zip(actual, predicted)])

# ────────────────────────────────────
# 7. ANSWER THE 10 QUESTIONS
# ────────────────────────────────────

# Q1: Row 25, DeBERTa alone
row25 = test_df.iloc[25]
text25 = row25["text"]
deberta_probs_25 = get_probabilities_batched(deberta_model, deberta_tokenizer, [text25])[0]
top1_idx = np.argmax(deberta_probs_25)
top1_letter = id2label[top1_idx]
top1_prob = deberta_probs_25[top1_idx]
print(f"Q1: {top1_letter}, {top1_prob:.4f}")

# Q2: Simple average ensemble, row 25
roberta_probs_25 = get_probabilities_batched(roberta_model, roberta_tokenizer, [text25])[0]
avg_probs = (deberta_probs_25 + roberta_probs_25) / 2.0
avg_top1_letter = id2label[np.argmax(avg_probs)]
print(f"Q2: {avg_top1_letter}")

# Q3: Weighted ensemble (0.7/0.3), row 25
weighted_probs_25 = 0.7 * deberta_probs_25 + 0.3 * roberta_probs_25
weighted_top1_letter = id2label[np.argmax(weighted_probs_25)]
print(f"Q3: {weighted_top1_letter}")

# Q4: Top‑3 prediction string, row 25
top3_str = " ".join(top_k_from_probs(weighted_probs_25, k=3))
print(f"Q4: {top3_str}")

# Q5: Number of rows in submission.csv
test_texts = test_df["text"].tolist()
print("Running DeBERTa inference on test set (batched)...")
deberta_all = get_probabilities_batched(deberta_model, deberta_tokenizer, test_texts, batch_size=16)
print("Running RoBERTa inference on test set (batched)...")
roberta_all = get_probabilities_batched(roberta_model, roberta_tokenizer, test_texts, batch_size=16)
weighted_all = 0.7 * deberta_all + 0.3 * roberta_all
all_top3 = [top_k_from_probs(p) for p in weighted_all]
submission = pd.DataFrame({
    "id": test_df["id"],
    "Prediction": [" ".join(t) for t in all_top3]
})
submission.to_csv("submission.csv", index=False)
print(f"Q5: {len(submission)} rows (excluding header)")

# Q6: TTA on first 50 rows
first50 = test_texts[:50]
deberta_probs_50 = get_probabilities_batched(deberta_model, deberta_tokenizer, first50, batch_size=16)
aug_texts = ["Answer the following multiple-choice question carefully: " + t for t in first50]
deberta_probs_50_aug = get_probabilities_batched(deberta_model, deberta_tokenizer, aug_texts, batch_size=16)
tta_probs = (deberta_probs_50 + deberta_probs_50_aug) / 2.0
orig_top1 = np.argmax(deberta_probs_50, axis=1)
tta_top1 = np.argmax(tta_probs, axis=1)
q6_diff = np.sum(orig_top1 != tta_top1)
print(f"Q6: {q6_diff} rows have different Top‑1 after TTA")

# Q7: Top‑1 differences between DeBERTa and Weighted Ensemble (first 100)
first100 = test_texts[:100]
deberta_probs_100 = get_probabilities_batched(deberta_model, deberta_tokenizer, first100, batch_size=16)
roberta_probs_100 = get_probabilities_batched(roberta_model, roberta_tokenizer, first100, batch_size=16)
weighted_probs_100 = 0.7 * deberta_probs_100 + 0.3 * roberta_probs_100
q7_diff = np.sum(np.argmax(deberta_probs_100, axis=1) != np.argmax(weighted_probs_100, axis=1))
print(f"Q7: {q7_diff} rows with different Top‑1 predictions")

# Q8: Positive confidence gain (first 100)
deberta_conf = np.max(deberta_probs_100, axis=1)
weighted_conf = np.max(weighted_probs_100, axis=1)
gain = weighted_conf - deberta_conf
q8_pos = np.sum(gain > 0)
print(f"Q8: {q8_pos} rows with positive confidence gain")

# Q9: Top‑3 ordering differences (first 100)
deberta_top3_100 = [top_k_from_probs(p) for p in deberta_probs_100]
weighted_top3_100 = [top_k_from_probs(p) for p in weighted_probs_100]
q9_diff = sum(1 for d, w in zip(deberta_top3_100, weighted_top3_100) if d != w)
print(f"Q9: {q9_diff} rows with at least one change in Top‑3 ordering")

# Q10: MAP@3 on first 100 validation samples
_, val_df = train_test_split(
    train_df, test_size=0.1, random_state=42, stratify=train_df["label"]
)
val_df = val_df.head(100)
val_texts = val_df["text"].tolist()
val_labels = val_df["answer"].tolist()

deberta_val_probs = get_probabilities_batched(deberta_model, deberta_tokenizer, val_texts, batch_size=16)
roberta_val_probs = get_probabilities_batched(roberta_model, roberta_tokenizer, val_texts, batch_size=16)
weighted_val_probs = 0.7 * deberta_val_probs + 0.3 * roberta_val_probs
val_top3 = [top_k_from_probs(p) for p in weighted_val_probs]

map3_val = mapk(val_labels, val_top3)
print(f"Q10: {map3_val:.4f}")

Removing previous checkpoint /kaggle/working/deberta_small_mcq
Fine‑tuning microsoft/deberta-v3-small ...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight     

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Q1: B, 0.2756
Q2: E
Q3: E
Q4: E B C
Running DeBERTa inference on test set (batched)...
Running RoBERTa inference on test set (batched)...
Q5: 500 rows (excluding header)
Q6: 0 rows have different Top‑1 after TTA
Q7: 74 rows with different Top‑1 predictions
Q8: 98 rows with positive confidence gain
Q9: 74 rows with at least one change in Top‑3 ordering
Q10: 0.9850


In [13]:
print(train.columns.tolist())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'label']


In [ ]:
from kaggle_secrets import UserSecretsClient
import wandb
import os

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login()
print("W&B logged in!")

In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import train_test_split
# from sklearn.pipeline import FeatureUnion

# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test_df  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
# OPTION_COLS = ["A", "B", "C", "D", "E"]

# def mapk(actual, predicted, k=3):
#     def apk(a, p):
#         score, hits = 0.0, 0
#         for i, pi in enumerate(p[:k]):
#             if pi == a:
#                 hits += 1
#                 score += hits / (i + 1)
#         return score
#     return np.mean([apk(a, p) for a, p in zip(actual, predicted)])

# # ── KEY IMPROVEMENT 1: Better text construction ───────────────
# # Include ALL options in context so model sees contrast
# def build_texts(df):
#     texts = []
#     for _, row in df.iterrows():
#         all_opts = " [OPT] ".join([str(row[c]) for c in OPTION_COLS])
#         for opt in OPTION_COLS:
#             combined = (
#                 str(row["prompt"]) + " " +
#                 str(row["prompt"]) +        # repeat prompt for weight
#                 " [SEP] " +
#                 str(row[opt]) +             # this specific option
#                 " [CONTEXT] " +
#                 all_opts                    # all options for contrast
#             )
#             texts.append(combined)
#     return texts

# train_texts = build_texts(train_df)
# test_texts  = build_texts(test_df)

# # ── KEY IMPROVEMENT 2: Better rank labels ─────────────────────
# # Instead of just 0 vs 4, use graded labels
# # Correct = 2, wrong = 0 (simpler but cleaner signal)
# label2idx = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

# train_labels = []
# for _, row in train_df.iterrows():
#     correct_idx = label2idx[row["answer"]]
#     for i in range(5):
#         train_labels.append(1 if i == correct_idx else 0)

# # ── KEY IMPROVEMENT 3: Better TF-IDF settings ────────────────
# word_tfidf = TfidfVectorizer(
#     analyzer="word",
#     ngram_range=(1, 3),
#     max_features=300000,      # more features
#     sublinear_tf=True,
#     strip_accents="unicode",
#     min_df=1
# )

# char_tfidf = TfidfVectorizer(
#     analyzer="char_wb",
#     ngram_range=(2, 5),       # start from 2 instead of 3
#     max_features=300000,
#     sublinear_tf=True,
#     strip_accents="unicode",
#     min_df=1
# )

# # ── KEY IMPROVEMENT 4: Add char-level word ngrams ─────────────
# char_word_tfidf = TfidfVectorizer(
#     analyzer="char",          # pure char (not word-boundary)
#     ngram_range=(3, 4),
#     max_features=200000,
#     sublinear_tf=True,
#     min_df=1
# )

# combined_features = FeatureUnion([
#     ("word", word_tfidf),
#     ("char_wb", char_tfidf),
#     ("char", char_word_tfidf),   # extra char features
# ])

# # ── Train/Val Split ───────────────────────────────────────────
# n = len(train_df)
# idx = np.arange(n)
# tr_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)

# tr_text_idx  = [i*5+j for i in tr_idx  for j in range(5)]
# val_text_idx = [i*5+j for i in val_idx for j in range(5)]

# tr_texts_split  = [train_texts[i] for i in tr_text_idx]
# val_texts_split = [train_texts[i] for i in val_text_idx]
# tr_labels_split = [train_labels[i] for i in tr_text_idx]

# print("Fitting TF-IDF (word + char_wb + char)...")
# X_train = combined_features.fit_transform(tr_texts_split)
# X_val   = combined_features.transform(val_texts_split)
# X_test  = combined_features.transform(test_texts)
# print(f"Feature matrix: {X_train.shape}")

# # ── KEY IMPROVEMENT 5: Train multiple C values, pick best ─────
# print("Training Logistic Regression...")
# best_map3, best_clf = 0, None

# for C in [0.5, 1.0, 3.0, 5.0, 10.0]:
#     clf = LogisticRegression(
#         C=C,
#         max_iter=2000,
#         solver="saga",
#         n_jobs=-1
#     )
#     clf.fit(X_train, tr_labels_split)
    
#     # Quick val check
#     probs = clf.predict_proba(X_val)
#     class1_idx = list(clf.classes_).index(1)
#     scores = probs[:, class1_idx].reshape(len(val_idx), 5)
#     top3 = np.argsort(-scores, axis=1)[:, :3]
#     preds = [[OPTION_COLS[i] for i in row] for row in top3]
#     true  = [train_df.iloc[i]["answer"] for i in val_idx]
#     m = mapk(true, preds)
#     print(f"  C={C} → MAP@3: {m:.4f}")
    
#     if m > best_map3:
#         best_map3 = m
#         best_clf  = clf

# print(f"\nBest Validation MAP@3: {best_map3:.4f}")

# # ── Final Predictions ─────────────────────────────────────────
# def predict(X, n_questions, clf):
#     probs = clf.predict_proba(X)
#     class1_idx = list(clf.classes_).index(1)
#     scores = probs[:, class1_idx].reshape(n_questions, 5)
#     top3 = np.argsort(-scores, axis=1)[:, :3]
#     return [[OPTION_COLS[i] for i in row] for row in top3]

# test_preds = predict(X_test, len(test_df), best_clf)

# submission = pd.DataFrame({
#     "id":         test_df["id"],
#     "Prediction": [" ".join(p) for p in test_preds]
# })
# submission.to_csv("submission.csv", index=False)
# print("Done!")
# print(submission.head(10))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# ── Load Data ──────────────────────────────────────────────────
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
OPTION_COLS = ["A", "B", "C", "D", "E"]

def mapk(actual, predicted, k=3):
    def apk(a, p):
        score, hits = 0.0, 0
        for i, pi in enumerate(p[:k]):
            if pi == a:
                hits += 1
                score += hits / (i + 1)
        return score
    return np.mean([apk(a, p) for a, p in zip(actual, predicted)])

# ── Build pairwise ranking features ───────────────────────────
# Key insight: instead of "is this option correct?" (binary)
# we ask "is option X better than option Y?" (ranking)
# This is what your friend did — rank-based not classify-based

def build_texts(df):
    """
    For each question, create 5 texts:
    prompt + option (word ngrams catch semantics,
    char ngrams catch subtle spelling/word differences)
    """
    texts = []
    for _, row in df.iterrows():
        for opt in OPTION_COLS:
            # Full text combination
            combined = (
                str(row["prompt"]) + " " +
                str(row["prompt"]) +  # repeat prompt to give it more weight
                " [SEP] " +
                str(row[opt])
            )
            texts.append(combined)
    return texts

train_texts = build_texts(train_df)
test_texts  = build_texts(test_df)

# ── Labels: rank-based (not binary 0/1) ───────────────────────
# Correct option gets score 4, others get 0,1,2,3 randomly
# This teaches the model to RANK not just classify
label2idx = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

train_labels = []
for _, row in train_df.iterrows():
    correct_idx = label2idx[row["answer"]]
    for i in range(5):
        # Correct = 4 (highest rank), wrong = 0
        train_labels.append(4 if i == correct_idx else 0)

# ── TF-IDF: BOTH word and character ngrams ─────────────────────
# Word ngrams: catch semantic meaning
# Char ngrams: catch subtle differences in similar paragraphs
# This is the KEY difference from basic TF-IDF

from sklearn.pipeline import Pipeline, FeatureUnion

word_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),      # unigrams, bigrams, trigrams
    max_features=200000,
    sublinear_tf=True,
    strip_accents="unicode",
    min_df=1
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",       # character ngrams within word boundaries
    ngram_range=(3, 5),       # 3,4,5 char ngrams
    max_features=200000,
    sublinear_tf=True,
    strip_accents="unicode",
    min_df=1
)

combined_features = FeatureUnion([
    ("word", word_tfidf),
    ("char", char_tfidf)
])

# ── Train/Val Split ────────────────────────────────────────────
n = len(train_df)
idx = np.arange(n)
tr_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)

tr_text_idx  = [i*5+j for i in tr_idx  for j in range(5)]
val_text_idx = [i*5+j for i in val_idx for j in range(5)]

tr_texts_split   = [train_texts[i] for i in tr_text_idx]
val_texts_split  = [train_texts[i] for i in val_text_idx]
tr_labels_split  = [train_labels[i] for i in tr_text_idx]

print("Fitting TF-IDF features (word + char ngrams)...")
X_train = combined_features.fit_transform(tr_texts_split)
X_val   = combined_features.transform(val_texts_split)
X_test  = combined_features.transform(test_texts)
print(f"Feature matrix shape: {X_train.shape}")

# ── Logistic Regression ────────────────────────────────────────
print("Training Logistic Regression...")
clf = LogisticRegression(
    C=5.0,
    max_iter=2000,
    solver="saga",
    n_jobs=-1
)
clf.fit(X_train, tr_labels_split)
print("Done!")

# ── Rank-based Prediction ──────────────────────────────────────
# Use predict_proba score for class 4 (correct option)
# This gives a continuous score per option → rank them

def get_scores_and_predict(X, n_questions):
    # Get probability of being the correct answer (class 4)
    probs = clf.predict_proba(X)
    # Find which column corresponds to class 4
    class4_idx = list(clf.classes_).index(4)
    scores = probs[:, class4_idx]
    # Reshape to (n_questions, 5)
    scores_matrix = scores.reshape(n_questions, 5)
    # Rank: top 3 per question
    top3 = np.argsort(-scores_matrix, axis=1)[:, :3]
    return [[OPTION_COLS[i] for i in row] for row in top3]

# Validate 
val_preds  = get_scores_and_predict(X_val, len(val_idx))
true_labels = [train_df.iloc[i]["answer"] for i in val_idx]
val_map3   = mapk(true_labels, val_preds)
print(f"\nValidation MAP@3: {val_map3:.4f}")

# Submission
test_preds = get_scores_and_predict(X_test, len(test_df))

submission = pd.DataFrame({
    "id":         test_df["id"],
    "Prediction": [" ".join(p) for p in test_preds]
})



submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
print(submission.head(10))

In [ ]:
print(submission.head(20))

In [ ]:
# Initialize W&B run
wandb.init(
    project="22f3001070-t22026",   
    name="tfidf-logreg-word-char-ngram"
)

# Log everything
wandb.log({
    "model": "TF-IDF + Logistic Regression",
    "word_ngram_range": "(1,3)",
    "char_ngram_range": "(3,5)",
    "max_features_word": 200000,
    "max_features_char": 200000,
    "C": 5.0,
    "solver": "saga",
    "label_type": "rank-based (0 vs 4)",
    "train_size": len(tr_idx) * 5,
    "val_size": len(val_idx) * 5,
    "val_map3": val_map3,
})

print(f"Logged to W&B — MAP@3: {val_map3:.4f}")
wandb.finish()

In [ ]:
# ##MILESTONE - 1

# import pandas as pd
# import numpy as np
# import string
# from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
# from sklearn.metrics.pairwise import cosine_similarity

# # Load the data
# train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# # ============================================
# # Question 1: Frequency distribution of correct answers
# # ============================================
# print("=" * 50)
# print("QUESTION 1: Frequency Distribution of Correct Answers")
# print("=" * 50)

# answer_freq = train_df['answer'].value_counts().sort_index()
# print("Frequency of each answer:")
# print(answer_freq)

# most_frequent = answer_freq.max()
# least_frequent = answer_freq.min()
# sum_most_least = most_frequent + least_frequent

# print(f"\nMost frequent option count: {most_frequent}")
# print(f"Least frequent option count: {least_frequent}")
# print(f"Sum of most and least frequent: {sum_most_least}")

# # ============================================
# # Question 2: Vocabulary size after cleaning prompts
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 2: Vocabulary Size After Cleaning")
# print("=" * 50)

# def clean_text(text):
#     # Convert to lowercase
#     text = text.lower()
#     # Remove punctuation
#     text = text.translate(str.maketrans('', '', string.punctuation))
#     return text

# # Clean all prompts
# train_df['cleaned_prompt'] = train_df['prompt'].apply(clean_text)

# # Get all unique words
# all_words = set()
# for prompt in train_df['cleaned_prompt']:
#     words = prompt.split()
#     all_words.update(words)

# vocab_size = len(all_words)
# print(f"Total unique words (vocabulary size): {vocab_size}")

# # ============================================
# # Question 3: Words left in Row ID 1 after removing stop words
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 3: Words in Row ID 1 After Removing Stop Words")
# print("=" * 50)

# # Get cleaned prompt for Row ID 1
# row1_prompt = train_df[train_df['id'] == 1]['cleaned_prompt'].values[0]

# # Split into words
# row1_words = row1_prompt.split()

# # Filter out stop words
# row1_filtered = [word for word in row1_words if word not in ENGLISH_STOP_WORDS]

# words_left = len(row1_filtered)
# print(f"Row ID 1 cleaned prompt (first 100 chars): {row1_prompt[:100]}...")
# print(f"Words left after removing stop words: {words_left}")

# # ============================================
# # Question 4: TF-IDF Vectorizer vocabulary size
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 4: TF-IDF Vectorizer Vocabulary Size")
# print("=" * 50)

# # Combine prompt and all options into single documents for each row
# combined_texts = []
# for idx, row in train_df.iterrows():
#     # Combine prompt with all options
#     combined = row['prompt'] + ' ' + row['A'] + ' ' + row['B'] + ' ' + row['C'] + ' ' + row['D'] + ' ' + row['E']
#     combined_texts.append(combined)

# # Fit TF-IDF vectorizer
# tfidf_vectorizer = TfidfVectorizer(stop_words='english')
# tfidf_vectorizer.fit(combined_texts)

# feature_columns = len(tfidf_vectorizer.get_feature_names_out())
# print(f"Number of feature columns (vocabulary size): {feature_columns}")

# # ============================================
# # Question 5: Cosine similarity between prompt and option A for Row ID 1
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 5: Cosine Similarity for Row ID 1 (Prompt vs Option A)")
# print("=" * 50)

# # Get Row ID 1 data
# row1 = train_df[train_df['id'] == 1].iloc[0]

# # Transform prompt and option A separately
# prompt_vector = tfidf_vectorizer.transform([row1['prompt']])
# option_a_vector = tfidf_vectorizer.transform([row1['A']])

# # Calculate cosine similarity
# similarity_score = cosine_similarity(prompt_vector, option_a_vector)[0][0]

# print(f"Prompt: {row1['prompt'][:100]}...")
# print(f"Option A: {row1['A'][:100]}...")
# print(f"Cosine similarity score: {similarity_score:.4f}")

# # ============================================
# # Question 6: Percentage where highest similarity matches correct answer
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 6: Percentage of Highest Similarity Matching Correct Answer")
# print("=" * 50)

# correct_matches = 0
# total_rows = len(train_df)

# for idx, row in train_df.iterrows():
#     # Vectorize prompt
#     prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
#     # Vectorize each option and calculate similarity
    
#     similarities = {}
#     for option in ['A', 'B', 'C', 'D', 'E']:
#         option_vec = tfidf_vectorizer.transform([row[option]])
#         sim = cosine_similarity(prompt_vec, option_vec)[0][0]
#         similarities[option] = sim
    
#     # Find option with highest similarity
#     highest_sim_option = max(similarities, key=similarities.get)
    
#     # Check if matches correct answer
#     if highest_sim_option == row['answer']:
#         correct_matches += 1

# percentage = (correct_matches / total_rows) * 100
# print(f"Rows where highest similarity matches correct answer: {correct_matches}/{total_rows}")
# print(f"Percentage: {percentage:.2f}%")

# # ============================================
# # Question 7: MAP@3 score for prediction C A B when answer is C
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 7: MAP@3 for prediction C A B (answer is C)")
# print("=" * 50)

# def calculate_map_at_3(ground_truth, predictions):
#     """
#     Calculate MAP@3 for a single question
#     predictions: list of 3 predicted answers in order
#     """
#     for i, pred in enumerate(predictions):
#         if pred == ground_truth:
#             return 1.0 / (i + 1)  # 1/k where k is the position (1-indexed)
#     return 0.0  # Not in top 3

# # Example: answer is C, prediction is C A B
# map_score_q7 = calculate_map_at_3('C', ['C', 'A', 'B'])
# print(f"Ground truth: C, Prediction: C A B")
# print(f"MAP@3 score: {map_score_q7}")

# # ============================================
# # Question 8: MAP@3 score for prediction D B E when answer is B
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 8: MAP@3 for prediction D B E (answer is B)")
# print("=" * 50)

# map_score_q8 = calculate_map_at_3('B', ['D', 'B', 'E'])
# print(f"Ground truth: B, Prediction: D B E")
# print(f"MAP@3 score: {map_score_q8}")

# # ============================================
# # Question 9: Majority Class Baseline MAP@3
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 9: Majority Class Baseline MAP@3")
# print("=" * 50)

# # Get frequency of answers
# answer_counts = train_df['answer'].value_counts()
# print("Answer frequencies:")
# print(answer_counts)

# # Get top 3 most frequent answers
# top3_answers = answer_counts.head(3).index.tolist()
# print(f"Top 3 most frequent answers: {top3_answers}")

# # Calculate MAP@3 for majority baseline
# majority_scores = []
# for idx, row in train_df.iterrows():
#     ground_truth = row['answer']
#     predictions = top3_answers  # Always predict the same top 3
#     score = calculate_map_at_3(ground_truth, predictions)
#     majority_scores.append(score)

# overall_majority_map = np.mean(majority_scores)
# print(f"Overall MAP@3 for Majority Class Baseline: {overall_majority_map:.4f}")

# # ============================================
# # Question 10: TF-IDF Pipeline MAP@3
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 10: TF-IDF Pipeline MAP@3")
# print("=" * 50)

# tfidf_scores = []

# for idx, row in train_df.iterrows():
#     # Vectorize prompt
#     prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
#     # Calculate similarity for each option
#     similarities = {}
#     for option in ['A', 'B', 'C', 'D', 'E']:
#         option_vec = tfidf_vectorizer.transform([row[option]])
#         sim = cosine_similarity(prompt_vec, option_vec)[0][0]
#         similarities[option] = sim
    
#     # Sort options by similarity (highest to lowest)
#     sorted_options = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    
#     # Get top 3 predictions
#     top3_predictions = [option for option, sim in sorted_options[:3]]
    
#     # Calculate MAP@3 for this row
#     ground_truth = row['answer']
#     score = calculate_map_at_3(ground_truth, top3_predictions)
#     tfidf_scores.append(score)

# overall_tfidf_map = np.mean(tfidf_scores)
# print(f"Overall MAP@3 for TF-IDF Pipeline: {overall_tfidf_map:.4f}")

# # ============================================
# # Create sample submission file
# # ============================================
# print("\n" + "=" * 50)
# print("Creating Sample Submission File")
# print("=" * 50)

# # Create predictions for test set using TF-IDF approach
# submission_predictions = []

# for idx, row in test_df.iterrows():
#     prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
#     similarities = {}
#     for option in ['A', 'B', 'C', 'D', 'E']:
#         option_vec = tfidf_vectorizer.transform([row[option]])
#         sim = cosine_similarity(prompt_vec, option_vec)[0][0]
#         similarities[option] = sim
    
#     sorted_options = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
#     top3_predictions = [option for option, sim in sorted_options[:3]]
    
#     submission_predictions.append({
#         'ID': row['id'],
#         'Prediction': ' '.join(top3_predictions)
#     })

# # Create submission DataFrame
# submission_df = pd.DataFrame(submission_predictions)

# # Save to CSV
# submission_df.to_csv('sample_submission.csv', index=False)

# print(f"Sample submission file created with {len(submission_df)} predictions")
# print("\nFirst few predictions:")
# print(submission_df.head())

In [ ]:
# # MILESTONE 2 
# !pip install -q transformers datasets sentence-transformers

# import torch
# import numpy as np
# import pandas as pd
# from datasets import load_dataset
# from transformers import AutoTokenizer, AutoModel, pipeline as hf_pipeline
# from sentence_transformers import SentenceTransformer, util
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

# # ── Load Data ────────────────────────────────────────────────
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# dataset  = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]
# OPTION_COLS = ["A", "B", "C", "D", "E"]

# def mapk(actual, predicted, k=3):
#     def apk(a, p):
#         score, hits = 0.0, 0
#         for i, pi in enumerate(p[:k]):
#             if pi == a:
#                 hits += 1
#                 score += hits / (i + 1)
#         return score
#     return np.mean([apk(a, p) for a, p in zip(actual, predicted)])

# # ── Q1: combined_text length at index 51 ─────────────────────
# dataset = dataset.map(lambda x: {"combined_text": x["prompt"] + " " + x["A"]})
# q1 = len(dataset[51]["combined_text"])
# print(f"Q1 - combined_text length at index 51: {q1}")

# # ── Q2: BERT vocab size ───────────────────────────────────────
# bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# q2 = bert_tokenizer.vocab_size
# print(f"Q2 - BERT vocab size: {q2}")

# # ── Q3: SEP token ID ─────────────────────────────────────────
# q3 = bert_tokenizer.sep_token_id
# print(f"Q3 - [SEP] token ID: {q3}")

# # ── Q4: Shape of input_ids tensor ────────────────────────────
# # ── Q4: Shape of input_ids tensor ────────────────────────────
# encoded = bert_tokenizer(
#     list(dataset["prompt"]),   # ← add list() here
#     padding="max_length",
#     truncation=True,
#     max_length=128,
#     return_tensors="pt"
# )
# q4 = tuple(encoded["input_ids"].shape)
# print(f"Q4 - input_ids shape: {q4}")

# # ── Q5: Attention head dimension ─────────────────────────────
# q5 = 768 // 12
# print(f"Q5 - Each attention head dimension: {q5}")

# # ── Q6: last_hidden_state shape for row 0 ────────────────────
# bert_model = AutoModel.from_pretrained("bert-base-uncased")
# bert_model.eval()
# inputs_row0 = bert_tokenizer(dataset[0]["prompt"], return_tensors="pt")
# with torch.no_grad():
#     outputs_row0 = bert_model(**inputs_row0)
# q6 = tuple(outputs_row0.last_hidden_state.shape)
# print(f"Q6 - last_hidden_state shape: {q6}")

# # ── Q7: Sum of first 5 CLS values ────────────────────────────
# cls_emb = outputs_row0.last_hidden_state[0, 0, :]
# q7 = round(cls_emb[:5].sum().item(), 4)
# print(f"Q7 - Sum of first 5 CLS values: {q7}")

# # ── Q8: Attention CLS→fusion in last layer, head 0 ───────────
# bert_attn = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
# bert_attn.eval()
# text = "Light-ion fusion is a technique."
# inputs_attn = bert_tokenizer(text, return_tensors="pt")
# with torch.no_grad():
#     outputs_attn = bert_attn(**inputs_attn)
# tokens = bert_tokenizer.convert_ids_to_tokens(inputs_attn["input_ids"][0])
# print(f"Tokens: {tokens}")
# fusion_idx = tokens.index("fusion")
# q8 = round(outputs_attn.attentions[-1][0, 0, 0, fusion_idx].item(), 4)
# print(f"Q8 - Attention CLS→fusion: {q8}")

# # ── Q9: Cosine sim prompt vs Option B row 0 ──────────────────
# st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# emb_prompt = st_model.encode(dataset[0]["prompt"], convert_to_tensor=True)
# emb_b      = st_model.encode(dataset[0]["B"],      convert_to_tensor=True)
# q9 = round(util.cos_sim(emb_prompt, emb_b).item(), 4)
# print(f"Q9 - Cosine similarity prompt vs B: {q9}")

# # ── Q10: MiniLM MAP@3 + count where MiniLM beats TF-IDF ──────
# def tfidf_predict(df):
#     preds = []
#     all_texts = df["prompt"].tolist() + df[OPTION_COLS].values.flatten().tolist()
#     vec = TfidfVectorizer(ngram_range=(1,2), max_features=50000)
#     vec.fit(all_texts)
#     for _, row in df.iterrows():
#         pv = vec.transform([row["prompt"]])
#         ov = vec.transform([row[c] for c in OPTION_COLS])
#         sims = cosine_similarity(pv, ov)[0]
#         top3 = [OPTION_COLS[i] for i in np.argsort(sims)[::-1][:3]]
#         preds.append(top3)
#     return preds

# def minilm_predict(df):
#     preds = []
#     for _, row in df.iterrows():
#         pemb  = st_model.encode(row["prompt"], convert_to_tensor=True)
#         oembs = st_model.encode([row[c] for c in OPTION_COLS], convert_to_tensor=True)
#         sims  = util.cos_sim(pemb, oembs)[0].cpu().numpy()
#         top3  = [OPTION_COLS[i] for i in np.argsort(sims)[::-1][:3]]
#         preds.append(top3)
#     return preds

# print("Running TF-IDF pipeline...")
# tfidf_preds  = tfidf_predict(train_df)
# print("Running MiniLM pipeline...")
# minilm_preds = minilm_predict(train_df)

# true_labels  = train_df["answer"].tolist()
# q10a = round(mapk(true_labels, minilm_preds), 4)
# print(f"Q10a - MiniLM MAP@3: {q10a}")

# q10b = sum(
#     1 for true, tp, mp in zip(true_labels, tfidf_preds, minilm_preds)
#     if true not in tp and true in mp
# )
# print(f"Q10b - Count MiniLM beats TF-IDF: {q10b}")

# # ── Q11: Zero-shot classification softmax ────────────────────
# zsc = hf_pipeline("zero-shot-classification")
# row1 = dataset[1]
# candidate_labels = [row1["A"], row1["B"], row1["C"]]
# result_softmax = zsc(row1["prompt"], candidate_labels)
# q11 = round(result_softmax["scores"][0], 4)
# print(f"Q11 - Top ranked score (softmax): {q11}")

# # ── Q12: Multi-label sigmoid vs softmax difference ───────────
# result_sigmoid = zsc(row1["prompt"], candidate_labels, multi_label=True)
# softmax_sum = sum(result_softmax["scores"])
# sigmoid_sum = sum(result_sigmoid["scores"])
# q12 = round(abs(softmax_sum - sigmoid_sum), 4)
# print(f"Q12 - Softmax sum: {round(softmax_sum,4)} | Sigmoid sum: {round(sigmoid_sum,4)}")
# print(f"Q12 - Absolute difference: {q12}")

# # ── Q13: Flan-T5 generative output ───────────────────────────
# from transformers import T5ForConditionalGeneration, T5Tokenizer

# flan_tok   = T5Tokenizer.from_pretrained("google/flan-t5-small")
# flan_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")
# flan_model.eval()

# row0 = dataset[0]
# input_str = (
#     f"Question: {row0['prompt']}. "
#     f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
#     f"Answer with just the letter A or B."
# )

# inputs = flan_tok(input_str, return_tensors="pt")
# with torch.no_grad():
#     out = flan_model.generate(**inputs, max_new_tokens=5)
# q13 = flan_tok.decode(out[0], skip_special_tokens=True)
# print(f"Q13 - Flan-T5 output: '{q13}'")
# # ── FINAL SUMMARY ─────────────────────────────────────────────
# print("\n========== ALL ANSWERS ==========")
# print(f"Q1  : {q1}")
# print(f"Q2  : {q2}")
# print(f"Q3  : {q3}")
# print(f"Q4  : {q4}")
# print(f"Q5  : {q5}")
# print(f"Q6  : {q6}")
# print(f"Q7  : {q7}")
# print(f"Q8  : {q8}")
# print(f"Q9  : {q9}")
# print(f"Q10a: {q10a}")
# print(f"Q10b: {q10b}")
# print(f"Q11 : {q11}")
# print(f"Q12 : {q12}")
# print(f"Q13 : {q13}")
# print("=================================")

In [ ]:
# """
# Milestone 3 
# """

# # ---------------------------------------------------------------------------
# # SETUP
# # ---------------------------------------------------------------------------
# !pip install faiss-cpu -q

# import pandas as pd
# import numpy as np
# import faiss
# from sentence_transformers import SentenceTransformer, CrossEncoder
# from transformers import AutoTokenizer, pipeline

# train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# print("Creating knowledge base")
# kb = []
# for idx, row in train.iterrows():
#     correct_letter = row['answer']
#     kb.append(str(row[correct_letter]))

# print("Loading embedding model and creating index")
# model = SentenceTransformer('all-MiniLM-L6-v2')
# kb_embeddings = model.encode(kb, show_progress_bar=True)
# index = faiss.IndexFlatL2(kb_embeddings.shape[1])
# index.add(kb_embeddings)
# print("Knowledge base successfully created:", len(kb), "documents")

# print("Loading zero-shot classifier")
# zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# print("Loading cross-encoder")
# cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# print("Loading tokenizer")
# tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# # Common row 150 setup
# row_150 = train.iloc[150]
# prompt_150 = str(row_150['prompt'])
# labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
# ans_150 = str(row_150[row_150['answer']])

# # ---------------------------------------------------------------------------
# # Q1 - Zero-shot classifier on row 150 (baseline, no RAG)
# # ---------------------------------------------------------------------------
# result_q1 = zs(prompt_150, candidate_labels=labels_150)
# score_q1 = result_q1['scores'][result_q1['labels'].index(ans_150)]
# print("\nQ1 answer:", round(score_q1, 3))

# # ---------------------------------------------------------------------------
# # Q2 - FAISS retrieval rank of true document for row 150
# # ---------------------------------------------------------------------------
# query_emb_150 = model.encode([prompt_150])
# distances, retrieved_indices = index.search(query_emb_150, 10)
# retrieved_indices = retrieved_indices[0]

# rank_q2 = list(retrieved_indices).index(150) + 1 if 150 in retrieved_indices else None
# print("Q2 answer (rank):", rank_q2)

# # ---------------------------------------------------------------------------
# # Q3 - Cross-encoder rerank rank of true document for row 150
# # ---------------------------------------------------------------------------
# docs_10 = [kb[i] for i in retrieved_indices]
# pairs = [[prompt_150, doc] for doc in docs_10]
# ce_scores = cross_encoder.predict(pairs)

# order = np.argsort(-ce_scores)
# sorted_kb_indices = [retrieved_indices[i] for i in order]
# rank_q3 = sorted_kb_indices.index(150) + 1 if 150 in sorted_kb_indices else None
# print("Q3 answer (rank):", rank_q3)

# # ---------------------------------------------------------------------------
# # Q4 - Token count for row 42 (top-5 retrieved docs, bert-base-uncased tokenizer)
# # ---------------------------------------------------------------------------
# row_42 = train.iloc[42]
# prompt_42 = str(row_42['prompt'])

# query_emb_42 = model.encode([prompt_42])
# _, ret_idx_42 = index.search(query_emb_42, 5)
# top5_docs_42 = [kb[i] for i in ret_idx_42[0]]

# concatenated_docs = " ".join(top5_docs_42)
# rag_string_42 = f"Context: {concatenated_docs} Question: {prompt_42}"

# num_tokens = len(tokenizer(rag_string_42, truncation=False)['input_ids'])
# print("Q4 answer (token count):", num_tokens)

# # ---------------------------------------------------------------------------
# # Q5 - Zero-shot with TRUE document as context (row 150)
# # ---------------------------------------------------------------------------
# true_doc_150 = kb[150]
# rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"

# result_q5 = zs(rag_string_150, candidate_labels=labels_150)
# score_q5 = result_q5['scores'][result_q5['labels'].index(ans_150)]
# print("Q5 answer:", round(score_q5, 3))

# # ---------------------------------------------------------------------------
# # Q6 - Adversarial RAG: force context = kb[999] (unrelated fact)
# # ---------------------------------------------------------------------------
# adversarial_doc = kb[999]
# adversarial_string = f"Context: {adversarial_doc} Question: {prompt_150}"

# result_q6 = zs(adversarial_string, candidate_labels=labels_150)
# score_q6 = result_q6['scores'][result_q6['labels'].index(ans_150)]
# print("Q6 answer:", round(score_q6, 3))

# # ---------------------------------------------------------------------------
# # Q7 - Hit Rate for first 100 rows (k=5)
# # ---------------------------------------------------------------------------
# hits = 0
# for idx in range(100):
#     row = train.iloc[idx]
#     prompt = str(row['prompt'])
#     correct_answer_text = str(row[row['answer']])

#     q_emb = model.encode([prompt])
#     _, top_idx = index.search(q_emb, 5)
#     retrieved_docs = [kb[i] for i in top_idx[0]]

#     if any(correct_answer_text in doc for doc in retrieved_docs):
#         hits += 1

# hit_rate = (hits / 100) * 100
# print("Q7 answer (hit rate %):", round(hit_rate, 1))

# # ---------------------------------------------------------------------------
# # Q8 - Full RAG pipeline (retrieve -> rerank -> augment -> predict) + MAP@3
# #      over first 20 rows
# # ---------------------------------------------------------------------------
# def average_precision_at_3(ranked_letters, correct_letter):
#     for i, letter in enumerate(ranked_letters[:3]):
#         if letter == correct_letter:
#             return 1.0 / (i + 1)
#     return 0.0

# ap_scores = []
# option_letters = ['A', 'B', 'C', 'D', 'E']

# for idx in range(20):
#     row = train.iloc[idx]
#     prompt = str(row['prompt'])
#     correct_letter = row['answer']
#     option_texts = [str(row[l]) for l in option_letters]

#     # Retrieve top 5
#     q_emb = model.encode([prompt])
#     _, top_idx = index.search(q_emb, 5)
#     top5_docs = [kb[i] for i in top_idx[0]]

#     # Rerank -> best single doc
#     pairs = [[prompt, doc] for doc in top5_docs]
#     scores = cross_encoder.predict(pairs)
#     best_doc = top5_docs[int(np.argmax(scores))]

#     # Augment
#     rag_string = f"Context: {best_doc} Question: {prompt}"

#     # Predict (pipeline already returns labels/scores sorted descending)
#     result = zs(rag_string, candidate_labels=option_texts)
#     label_to_letter = dict(zip(option_texts, option_letters))
#     ranked_letters = [label_to_letter[label] for label in result['labels']]

#     ap = average_precision_at_3(ranked_letters, correct_letter)
#     ap_scores.append(ap)
#     print(f"Row {idx}: correct={correct_letter}, top3={ranked_letters[:3]}, AP@3={ap}")

# map_at_3 = np.mean(ap_scores)
# print("\nQ8 answer (MAP@3):", round(map_at_3, 3))

# # ---------------------------------------------------------------------------
# # SUMMARY
# # ---------------------------------------------------------------------------
# print("\n===== SUMMARY =====")
# print("Q1:", round(score_q1, 3))
# print("Q2:", rank_q2)
# print("Q3:", rank_q3)
# print("Q4:", num_tokens)
# print("Q5:", round(score_q5, 3))
# print("Q6:", round(score_q6, 3))
# print("Q7:", round(hit_rate, 1))
# print("Q8:", round(map_at_3, 3))

In [ ]:
#Milestone-4

# import pandas as pd
# import torch
# from datasets import Dataset
# from transformers import (
#     AutoTokenizer,
#     AutoModelForMultipleChoice,
#     TrainingArguments,
#     Trainer
# )
# from peft import LoraConfig, TaskType, get_peft_model
# import numpy as np

# train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")


# # Q1. Label Encoding
# # Convert the answer column in train.csv into numeric labels using the following mapping:
# # A = 0
# # B = 1
# # C = 2
# # D = 3
# # E = 4

# # What is the encoded numeric label for the row at index 150?

# label_map = {'A':0,'B':1,'C':2,'D':3,'E':4}
# train["label"] = train.answer.map(label_map)

# print("Q1 =", train.loc[150,"label"])

# # Q2. Prompt-Option Formatting
# # For row index 0, create the Option B input using exactly this format:
# # str(prompt) + " [SEP] " + str(option_B)

# # What is the exact character length of this formatted input string?

# formatted = str(train.loc[0, "prompt"]) + " [SEP] " + str(train.loc[0, "B"])

# print("Q2 =", len(formatted))


# # Tokenization for Multiple-Choice Models
# # Multiple-choice models expect inputs in the shape:
# # batch_size x num_choices x sequence_length

# # Since each question has five options, every row becomes five tokenized sequences.
# # Q3. Single-Row MCQ Tokenization
# # Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
# # padding = "max_length"
# # truncation = True
# # max_length = 128
# # return_tensors = "pt"

# # After reshaping for a multiple-choice model, the final input_ids tensor has shape:
# # [1, 5, 128]

# # What is the value of the second dimension?

# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# choices = [
#     str(train.loc[0,"prompt"]) + " [SEP] " + str(train.loc[0,c])
#     for c in "ABCDE"
# ]

# enc = tokenizer(
#     choices,
#     padding="max_length",
#     truncation=True,
#     max_length=128,
#     return_tensors="pt"
# )

# enc = {k:v.unsqueeze(0) for k,v in enc.items()}

# print("Q3 =", enc["input_ids"].shape)




# # Q4. Batch MCQ Tokenization
# # Tokenize the first 16 rows of train.csv as multiple-choice examples.
# # Each row has 5 choices.
# # Each choice is tokenized to length 128.

# # The final input_ids tensor has shape:
# # [16, 5, 128]

# # How many total token positions are in this tensor?

# batch_choices = []
# for i in range(16):
#     batch_choices.append([
#         str(train.loc[i,"prompt"]) + " [SEP] " + str(train.loc[i,c])
#         for c in "ABCDE"
#     ])

# flat = [x for row in batch_choices for x in row]

# tok = tokenizer(
#     flat,
#     padding="max_length",
#     truncation=True,
#     max_length=128,
#     return_tensors="pt"
# )

# input_ids = tok["input_ids"].view(16,5,128)


# print("Q4 =", input_ids.numel())



# # Q5. Multiple-Choice Logits
# # Load bert-base-uncased using AutoModelForMultipleChoice.
# # Tokenize row index 0 as 5 choices and pass it through the model.

# # The output logits tensor has shape:
# # [1, 5]

# # How many logits are produced for one question?

# model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# out = model(**enc)

# print("Q5 =", out.logits.shape[1])



# # Q6. Supervised Loss Tensor
# # For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

# # The model returns a scalar loss tensor.

# # How many dimensions does this loss tensor have?

# labels = torch.tensor([train.loc[0,"label"]])

# loss = model(**enc, labels=labels).loss

# print("Q6 =", loss.dim())




# # LoRA for Efficient Fine-Tuning
# # LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.

# # Q7. LoRA Trainable Parameters
# # Apply LoRA to the bert-base-uncased multiple-choice model using:
# # r = 8
# # lora_alpha = 16
# # target_modules = ["query", "value"]
# # lora_dropout = 0.1
# # bias = "none"
# # task_type = TaskType.SEQ_CLS

# # Count trainable parameters using:
# # sum(p.numel() for p in model.parameters() if p.requires_grad)

# # How many parameters are trainable?

# config = LoraConfig(
#     r=8,
#     lora_alpha=16,
#     target_modules=["query","value"],
#     lora_dropout=0.1,
#     bias="none",
#     task_type=TaskType.SEQ_CLS,
# )

# model = get_peft_model(model, config)

# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

# print("Q7 =", trainable)



# # Preparing Data for Hugging Face Trainer
# # Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.
# # Q8. Hugging Face Dataset Preparation
# # Create a Hugging Face Dataset from the first 100 rows of train.csv.

# # For each row, create:
# # input_ids with shape [5, 128]
# # attention_mask with shape [5, 128]
# # labels as the encoded answer label

# # For the first dataset item, input_ids has shape:
# # [5, 128]

# # How many tokenized choices are stored in input_ids?

# dataset = Dataset.from_pandas(train.iloc[:100])

# def preprocess(example):
#     texts = [
#         str(example["prompt"]) + " [SEP] " + str(example[c])
#         for c in "ABCDE"
#     ]
#     tok = tokenizer(
#         texts,
#         max_length=128,
#         truncation=True,
#         padding="max_length",
#     )
#     tok["labels"] = label_map[example["answer"]]
#     return tok

# dataset = dataset.map(preprocess)

# print("Q8 =", len(dataset[0]["input_ids"]))



# # Tiny Fine-Tuning and Inference
# # In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.
# # Q9. Tiny LoRA Fine-Tuning
# # Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

# # Use the following settings:
# # max_length = 64
# # per_device_train_batch_size = 4
# # gradient_accumulation_steps = 1
# # max_steps = 4

# # What is the final global_step reported by the Trainer?

# small = train.iloc[:32]
# dataset = Dataset.from_pandas(small)
# dataset = dataset.map(preprocess)

# dataset.set_format(type="torch")

# args = TrainingArguments(
#     output_dir="tmp",
#     max_steps=4,
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=1,
#     report_to="none",
# )

# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=dataset,
# )

# trainer.train()

# print("Q9 =", trainer.state.global_step)



# # Q10. Probability Assigned to Option E After Fine-Tuning
# # Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

# # What is the probability assigned to Option E?

# # Round your answer to 4 decimal places.


# choices = [
#     str(train.loc[0,"prompt"]) + " [SEP] " + str(train.loc[0,c])
#     for c in "ABCDE"
# ]

# tok = tokenizer(
#     choices,
#     return_tensors="pt",
#     padding="max_length",
#     truncation=True,
#     max_length=64
# )

# tok = {k:v.unsqueeze(0) for k,v in tok.items()}

# device = next(model.parameters()).device

# tok = {k: v.to(device) for k, v in tok.items()}

# model.eval()
# with torch.no_grad():
#     logits = model(**tok).logits

# probs = torch.softmax(logits, dim=-1)

# print("Q10 =", round(probs[0, 4].item(), 4))
